# Notebook to play with option contracts

Latest version: 2026-08-05  
Author: MvS

## Description

This is an interactive, stateful option trading guidance system.

Use case: user wants to trade either cash-secured puts (CSP), or more advanced, poor-man's covered calls (PMCC).
0. the trading guidance should be running be calling notebook cells, maybe repeatingly, remembering the current state of trading affairs,
getting user input on bought or sold contracts.
1. a list of valid names/symbols for trading need to be established and a screening for general market sentiment needs to be collected.
only symbols which exceed a certain user-defined threshold should be open for trading.
2. the users portfolio of active and expired trades, i.e., order book w.r.t. option trading should be manageable by calling a current
stat export function: which manages an offline editing a CSV or excel file - eventually moving to db tables.
3. the system can pull current market data from yahoo finance to get an impression of current underlying, strike, expiration pricing and volatility  
4. the system should be able to suggest either CSP with an added a de-risking PUT option bought or a PMCC trading setups. Only whitelisted symbols can qualify, see, item 1
The setups should naturally qualify in terms of investment risk, volatility guard rails, risk level.
5. the user can confirm that he entered into contracts following any of the given suggestions, which enters them into his order book
6. there should be a possibility to evaluate the users running trade setups that move closer to expiration for de-risking suggestions:
    - keep open,
    - roll,
    - close for damage control,
    - or close for sufficient realized gains
The system reflects these changes in the order book.

Hint:
- use provided excel of a typical order book for basic design, [here](../data/Order_book_template.xlsx)
- we should optimize the excel template for option trading attributes
- use discussion about option strategies for [background info](../data/background_information.txt) and add own research, if required.
- use yahoo finance (YF) endpoint to access relevant trading data, e.g. option grids PUT/CALL, Strike, duration, bid, ask, theta, see next cells for example code which should be improved.
- use NVIDIA as a starting name
- use wall street as market
- use stock option symbol name in YF nomenclature, i.e.: `<SYMBOL><DATE:YYMMDD><TYPE:C/P><STRIKEPRICE*1000:08.0f>`  
  ToDo: find proper symbol template for YF to safely identify options
- in case relevant greeks are missing from market data: add methods to calculate them: delta, theta, etc... Use a preexisting lightweight library 
- always clarify open issues

Clarifications:
- Q: Do you want the order book to remain in Excel, or should we use a simpler notebook-native format like CSV/JSON/Parquet?  
  A: We can also use CSVs as long as the format can easily be imported to excel
- Q: Should we prioritize PMCC only, or include PMCSP and strangles in the first pass?  
  A: We can prioritize PMCCs
- Q: How strict should the whitelist be: explicit symbols only, or dynamic scanning from a candidate list?  
  A: I expect to do symbol evaluation only once a month from 10-30 symbols - hand-picked or suggested later maybe from you own research - which can enter or drop out of a number of  trading cycles
- Q: Should the notebook enforce NYSE trading hours for live option-data pulls?  
  A: Yes it should only pull during trading hours
- Q: Do you want the notebook to compute Greeks itself if Yahoo data lacks them, or just use available fields?  
  A: Yes we will need the greeks, I gues you find the formulas or use another python package fo that


In [ ]:
import numpy as np

def american_option_tree(S, K, T, r, sigma, option_type='call', steps=100):
    """
    Prices an American option using the Cox-Ross-Rubinstein Binomial Tree.
    S: Stock price, K: Strike, T: Time to maturity (years), r: Risk-free rate, sigma: Volatility
    """
    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    df = np.exp(-r * dt)

    # Initialize asset prices at maturity
    fs = np.zeros(steps + 1)
    S_t = S * (u ** np.arange(steps, -1, -1)) * (d ** np.arange(0, steps + 1))

    # Initialize option values at maturity
    if option_type == 'call':
        fs = np.maximum(S_t - K, 0)
    else:
        fs = np.maximum(K - S_t, 0)

    # Step backward through the tree
    for i in range(steps - 1, -1, -1):
        S_t = S * (u ** np.arange(i, -1, -1)) * (d ** np.arange(0, i + 1))

        # Value if we hold the option
        hold_value = df * (p * fs[:-1] + (1 - p) * fs[1:])

            # Value if we exercise early
        if option_type == 'call':
            exercise_value = np.maximum(S_t - K, 0)
        else:
            exercise_value = np.maximum(K - S_t, 0)

        # American condition: take the maximum of holding vs exercising
        fs = np.maximum(hold_value, exercise_value)

    return fs[0]

In [ ]:
import yfinance as yf
import datetime
from math import log2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from importlib import reload  # Python 3.4+
import utils.indicator_utils as idc

import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(message)s")

# Standard SMAs
periods = [200, 50]

# stock = "FROG"
# stock = "TSLA"
stock = "NVDA"
# stock = "SAP.TO"
# stock = "GLEN.L"
#stock = "ADM.L"
# stock = "SPX.L"
# stock = "JPM"
#stock = "KDP"


dt_end = datetime.datetime.today()
# Define real-time interval:
#  - assume to display at least the number of sample points of the larger period
#  - this requires double the number of points to create the averaging
#  - plus considering non-trading days - yfinance returns only trading days, howevers
dt_data_start = dt_end - datetime.timedelta(days=max(periods) * 3)


def fetch_stock_data(symbol, start, end, max_retries=2):
    start_date = start.strftime('%Y-%m-%d') if isinstance(start, datetime.datetime) else start
    end_date = end.strftime('%Y-%m-%d') if isinstance(end, datetime.datetime) else end

    for attempt in range(1, max_retries + 1):
        logging.info('Downloading %s attempt %s/%s', symbol, attempt, max_retries)
        try:
            df = yf.download(
                symbol,
                start=start_date,
                end=end_date,
                progress=False,
                threads=False,
            )
            if df is None or df.empty:
                logging.warning('yf.download returned no data for %s on attempt %s', symbol, attempt)
                continue
            return df
        except Exception as exc:
            logging.warning('yf.download failed for %s on attempt %s: %s', symbol, attempt, exc)

    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(start=start_date, end=end_date, interval='1d')
        if df is None or df.empty:
            raise ValueError('Ticker.history returned no data')
        return df
    except Exception as exc:
        raise RuntimeError(f'Unable to load data for {symbol}: {exc}')


try:
    load_df = fetch_stock_data(stock, dt_data_start, dt_end)
    if load_df.shape[0] <= max(periods):
        raise ValueError(
            f'Insufficient rows ({load_df.shape[0]}) for {stock}; need more than {max(periods)}'
        )
    if 'Adj Close' in load_df.columns and 'Close' not in load_df.columns:
        load_df['Close'] = load_df['Adj Close']

    logging.info('Loaded %s rows for %s with columns %s', load_df.shape[0], stock, list(load_df.columns))

except Exception as exc:
    logging.info('Download failed for symbol %s. Skipping... %s', stock, exc)
    load_df = pd.DataFrame()


### Load and normalize the option order book
This cell loads the `Options` sheet from `data/Order_Book_template.xlsx` and creates a clean `position_df` that can be used for option-trade analysis.

In [ ]:
order_book_path = "../data/Order_Book_template.xlsx"
order_book_sheet = "Options"

# Read the Options sheet using the row with field names as header.
pos_df = pd.read_excel(order_book_path, sheet_name=order_book_sheet, header=3)

# Normalize and clean column names.
pos_df.columns = [
    str(c).strip().replace("\n", " ").replace("  ", " ")
    if not pd.isna(c)
    else ""
    for c in pos_df.columns
]
pos_df = pos_df.loc[:, [col for col in pos_df.columns if col]]

# Convert date columns to datetime.
date_cols = [col for col in pos_df.columns if "date" in col.lower()]
for col in date_cols:
    pos_df[col] = pd.to_datetime(pos_df[col], errors="coerce")

# Ensure duplicate columns are removed cleanly.
if pos_df.columns.duplicated().any():
    pos_df = pos_df.loc[:, ~pos_df.columns.duplicated()]

print(f"Loaded order book positions: {pos_df.shape}")
print(pos_df.columns.tolist())

display(pos_df.head(5))

position_df = pos_df.copy()
position_df["Status"] = position_df["Status"].astype(str).str.upper()
for numeric_col in ["Open price", "Lot price", "Open quantity", "Lot quantity"]:
    if numeric_col in position_df.columns:
        position_df[numeric_col] = pd.to_numeric(position_df[numeric_col], errors="coerce")

open_options = position_df[
    position_df["Status"].eq("OPEN")
    & position_df["Equity"].astype(str).str.contains("OPTION", na=False)
]
print(f"Open option positions: {len(open_options)}")
display(open_options)

### Definiton and calculation of ATR indicator

Source: [Investopedia](https://www.investopedia.com/terms/a/atr.asp)

1. True range is defined as `TR = Max[(H−L), ∣H−Cp​∣, ∣L−Cp​∣]` where:

    - `H`: Today’s high
    - `L`: Today’s low
    - `Cp`: Yesterday’s closing price
    - `Max`: Highest value of the three terms

2. ATR is the arithmetic mean of the daily true ranges over previous periods

3. Added a low-end volatility estimate which I define as `LR = Max[((Cp+H)/2.0) - L, (Cp - L)]`


In [ ]:
reload(idc)

# Now that we calculated the SMAs and EMAs, we can remove the data points before the actual AOI that we want.
stock_df = load_df[-max(periods) :].copy()


for opt in ['TR', 'LR', 'HR']:
    stock_df[opt] = idc.vola_range(
        close=stock_df['Close'],
        high=stock_df['High'],
        low=stock_df['Low'],
        mode=f"A{opt}",
        length=1
    )

# Define the columns and their styles
columns = [
    'High',
    'Low',
    'Close',
    'HR',
    'LR',
]
colors = ['darkgreen', 'red', 'forestgreen', 'grey', 'red']
linestyles = ['-', '-', ':', '-', '-']

# Plot each column separately
ax = None
for col, color, style in zip(columns, colors, linestyles):
    ax = stock_df[col].plot(color=color, linestyle=style, ax=ax)

ax.legend(loc='upper left')

# clean up
stock_df.drop(
    ['ClosePrev'],
    axis=1,
    inplace=True,
    errors='ignore',
)

### Plotting the Chart

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create a subplot with secondary y-axis enabled
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add the candlestick chart
fig.add_trace(
    go.Candlestick(
        x=stock_df.index,
        open=stock_df['Open'],
        high=stock_df['High'],
        low=stock_df['Low'],
        close=stock_df['Close'],
        name=f"{stock}",
    ),
    secondary_y=False,  # Assign to the primary y-axis
)

# Add the scatter plot for negative volatility
fig.add_trace(
    go.Scatter(
        x=stock_df.index,
        y=stock_df[f"LR"],
        mode='lines',
        name=f"Negative volatility",
        line=dict(color='#FF5252', width=2, dash='dot'),
    ),
    secondary_y=True,  # Assign to the secondary y-axis
)

# Update y-axes to use a logarithmic scale (optional)
fig.update_yaxes(type='log', secondary_y=False)  # Log scale for primary y-axis
fig.update_yaxes(title_text="Negative Volatility", secondary_y=True)

# Update layout
fig.update_layout(
    title='Standard candlesticks and trading signals based on EMAs',
    yaxis_title=f"{stock} Stock",
    xaxis_rangeslider_visible=False,
    width=1200,
    height=800,
)

# Show the plot
fig.show()

### Identify extrema: full ATR

In [ ]:
outliers = int(len(stock_df) * 0.05)

# Your original plot
ax = stock_df['TR'].plot(label='ATR', linestyle='-')

# Get the indices of the top 10 values in 'TR'
markers_on = stock_df['TR'].sort_values(ascending=False)[:outliers]

label = f'top {outliers} extrema'

# Loop over the selected indices and plot markers
for index in markers_on.index:
    ax.plot(index, stock_df['TR'].loc[index], 'oy', mfc='none', label=label)
    label = None

plt.legend()
plt.show()

### Identify extrema: negative range

In [ ]:
outliers = int(len(stock_df) * 0.05)

# Your original plot
ax = stock_df['LR'].plot(label='line', linestyle='-')

# Get the indices of the top 10 values in 'TR'
markers_series = stock_df['LR'].sort_values(ascending=False)[:outliers]

label = f'top {outliers} extrema'

# Loop over the selected indices and plot markers
for index in markers_series.index:
    ax.plot(index, stock_df['LR'].loc[index], 'oy', mfc='none', label=label)
    label = None

plt.legend()
plt.show()

### Identify extrema: positive range

In [ ]:
outliers = int(len(stock_df) * 0.05)

# Your original plot
ax = stock_df['HR'].plot(label='line', linestyle='-')

# Get the indices of the top 10 values in 'TR'
markers_series = stock_df['HR'].sort_values(ascending=False)[:outliers]

label = f'top {outliers} extrema'

# Loop over the selected indices and plot markers
for index in markers_series.index:
    ax.plot(index, stock_df['HR'].loc[index], 'oy', mfc='none', label=label)
    label = None

plt.legend()
plt.show()

### Extract isolated maxima from series

To get an estimate for the strongest daily movement in a turbulent trading phase

Note, that this extremum and neigboring suppressed extrema can accumulate to much larger numbers.

In [ ]:
reload(idc)

high_markers_series = idc.find_isolated_spikes(stock_df['HR'], num_spikes=10, n_distance=5)
low_markers_series = idc.find_isolated_spikes(stock_df['LR'], num_spikes=10, n_distance=5)

# Merge series to df
extrema_df = pd.concat(
    [high_markers_series, low_markers_series], axis=1, keys=['HR', 'LR']
)
extrema_df.sort_index(ascending=True, axis=0, inplace=True)

# Plot extrema
fig, axes = plt.subplots(nrows=2, ncols=1)

# Your original plot
ax_hr = stock_df['HR'].plot(ax=axes[0], label='HR line', linestyle='-')
ax_lr = stock_df['LR'].plot(ax=axes[1], label='LR line', linestyle='-')

label = f'top {outliers} iso-extrema'

# Loop over the selected indices and plot markers
for index in extrema_df[~extrema_df['HR'].isna()].index:
    ax_hr.plot(index, stock_df['HR'].loc[index], 'oy', mfc='none', label=label)
    label = None

label = f'top {outliers} iso-extrema'
for index in extrema_df[~extrema_df['LR'].isna()].index:
    ax_lr.plot(index, stock_df['LR'].loc[index], 'oy', mfc='none', label=label)
    label = None

fig.set_figwidth(12)
fig.set_figheight(8)
ax_hr.legend(loc="upper left")
ax_lr.legend(loc="upper left")

# plt.figure(figsize=(12,6))
# plt.legend()
plt.show()

del high_markers_series, low_markers_series

### Calculate different averages

In [ ]:
##---- Different means
scaler = 1.0

# Take valid extrema
for item in ['HR', 'LR']:
    markers_series = extrema_df[~extrema_df[item].isna()][item]

    arithmetic_mean = markers_series.sum() / len(markers_series) * scaler
    harmonic_mean = len(markers_series) / (1 / markers_series).sum() * scaler
    geometric_mean = np.exp(np.log(markers_series).mean()) * scaler

    logging.info(
        f""" {item} extrema:
    {'Arithmetic mean' : <16}: {arithmetic_mean:8.2f}
    {'Harmonic mean' : <16}: {harmonic_mean:8.2f}
    {'Geometric mean' : <16}: {geometric_mean:8.2f}
    {'Extrema' : <16}: {extrema_df[item].nlargest(3).to_list()}
    """
    )

In [ ]:
raise Exception('Stopping Here')

### Run iterative analysis of volatility for many symbols

In [ ]:
import yfinance as yf
import datetime
import pandas as pd
import numpy as np

import utils.indicator_utils as idc

import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(message)s")

# Standard SMAs
periods = [200, 50]

# CapTrader - USD
stocks = ['AAPL', 'COST', 'HLT', 'HSY', 'MCD',
    'MDLZ', 'MSFT', 'NVDA', 'PEP', 'SCI', 'WM',
    'SJM', 'V', 'JPM', 'GOOGL', 'AMZN'
]

# CapTrader - DE
# stocks = ['FRE.DE', 'DHL.DE', 'TDIV.AS', 'IBC1.MU', 'VDCA.MI']
# #stock = 'SC0Y.MU' # no chart


dt_end = datetime.datetime.today()
# Define real-time interval:
#  - assume to display at least the number of sample points of the larger period
#  - this requires double the number of points to create the averaging
#  - plus considering non-trading days - yfinance returns only trading days, howevers
dt_data_start = dt_end - datetime.timedelta(days=max(periods) * 3)

result_dict = {}

for stock in stocks:

    try:
        # Grab sufficient stock data for averaging SMAs
        load_df = yf.download(
            f"{stock}",
            start=dt_data_start.strftime('%Y-%m-%d'),
            end=dt_end.strftime('%Y-%m-%d'),
            progress=False,
        )

        assert load_df.shape[1] == 6 and load_df.shape[0] > max(periods)
    except AssertionError:
        logging.info(f"Download failed for symbol {stock}.  Skipping...")
    stock_df = load_df[-max(periods) :].copy()


    for opt in ['TR', 'LR', 'HR']:
        stock_df[opt] = idc.vola_range(
            close=stock_df['Close'],
            high=stock_df['High'],
            low=stock_df['Low'],
            mode=f"A{opt}",
            length=1
        )
    outliers = 10

    high_markers_series = idc.find_isolated_spikes(stock_df['HR'], num_spikes=outliers, n_distance=5)
    low_markers_series = idc.find_isolated_spikes(stock_df['LR'], num_spikes=outliers, n_distance=5)

    del stock_df

    # Merge series to df
    extrema_df = pd.concat(
        [high_markers_series, low_markers_series], axis=1, keys=['HR', 'LR']
    )
    extrema_df.sort_index(ascending=True, axis=0, inplace=True)

    del high_markers_series, low_markers_series

    # Take valid extrema
    for item in ['HR', 'LR']:
        markers_series = extrema_df[~extrema_df[item].isna()][item]

        arithmetic_mean = markers_series.sum() / len(markers_series)
        harmonic_mean = len(markers_series) / (1 / markers_series).sum()
        geometric_mean = np.exp(np.log(markers_series).mean())
        highest = extrema_df[item].nlargest(3).to_list()
        most_likely = max(arithmetic_mean, harmonic_mean, geometric_mean, highest[2])
        logging.info(
            f""" {item} extrema:
        {'Arithmetic mean' : <16}: {arithmetic_mean:8.2f}
        {'Harmonic mean' : <16}: {harmonic_mean:8.2f}
        {'Geometric mean' : <16}: {geometric_mean:8.2f}
        {'Extrema' : <16}: {highest}
        {'Most likely' : <16}: {most_likely:8.2f}
        """
        )
        if item == 'LR':
            result_dict[stock] = (highest[0], highest[1], most_likely)

    del extrema_df

# display results
for key, value in result_dict.items():
    print(key, value[2])
